# Notebook III-c: Diffusion-Based PET Reconstruction

From noise to medical images: how denoising diffusion probabilistic models (DDPM) can reconstruct PET scans — with uncertainty quantification for free.

**What you'll learn:**
- DDPM fundamentals: forward/reverse diffusion
- Time-conditioned U-Net architecture
- Training a diffusion model on synthetic PET data
- Measurement-consistent sampling (posterior guidance)
- Comparison: OSEM vs GAN vs Diffusion
- Uncertainty maps from multiple samples

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Why Diffusion for Medical Imaging?

PET reconstruction is an **inverse problem**: we observe noisy, incomplete measurements (sinograms) and want to recover the underlying tracer distribution (image). Several approaches exist:

| Method | Strengths | Weaknesses |
|--------|-----------|------------|
| **OSEM** | Physics-based, well-understood, no training data | Noisy at low counts, slow convergence |
| **GANs** | Sharp outputs, fast inference | Mode collapse, hallucination risk, no uncertainty |
| **Diffusion** | Stable training, principled probabilistic framework | Slow sampling (being addressed) |

**Key advantages of diffusion models:**

1. **No mode collapse**: Unlike GANs, diffusion models don't suffer from training instability or mode dropping.
2. **Uncertainty quantification for free**: Run the reverse process multiple times $\rightarrow$ get a distribution of reconstructions $\rightarrow$ pixel-wise uncertainty maps.
3. **Measurement consistency without retraining**: Condition the reverse process on measurements at inference time — no need to retrain for different noise levels or scanner geometries.
4. **Principled probabilistic framework**: The model learns the data distribution $p(x)$, and we can combine it with the likelihood $p(y|x)$ via Bayes' rule.

## DDPM Fundamentals

### Forward Process (Adding Noise)

Gradually add Gaussian noise over $T$ steps:

$$q(x_t | x_{t-1}) = \mathcal{N}\left(\sqrt{1 - \beta_t}\, x_{t-1},\; \beta_t \mathbf{I}\right)$$

where $\beta_1 < \beta_2 < \cdots < \beta_T$ is a **noise schedule** (linear from $10^{-4}$ to $0.02$).

After $T$ steps: $x_T \approx \mathcal{N}(0, \mathbf{I})$ — pure noise.

**Closed-form sampling at any step** (no need to iterate):

$$q(x_t | x_0) = \mathcal{N}\left(\sqrt{\bar{\alpha}_t}\, x_0,\; (1 - \bar{\alpha}_t) \mathbf{I}\right)$$

where $\bar{\alpha}_t = \prod_{s=1}^{t} (1 - \beta_s)$.

### Reverse Process (Denoising)

Learn to reverse the forward process:

$$p_\theta(x_{t-1} | x_t) = \mathcal{N}\left(\mu_\theta(x_t, t),\; \sigma_t^2 \mathbf{I}\right)$$

**Training objective**: predict the noise $\epsilon$ that was added:

$$\mathcal{L} = \mathbb{E}_{x_0, \epsilon, t} \left[\| \epsilon - \epsilon_\theta(\sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \epsilon,\; t) \|^2\right]$$

This is remarkably simple: sample a clean image, add noise, predict the noise, take MSE loss.

In [ ]:
# ============================================================
# Noise Schedule
# ============================================================

def linear_beta_schedule(timesteps, beta_start=1e-4, beta_end=0.02):
    """Linear noise schedule from beta_start to beta_end."""
    return torch.linspace(beta_start, beta_end, timesteps)

T = 300  # fewer steps for tractability on CPU
betas = linear_beta_schedule(T)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

# For posterior variance (used in reverse process)
alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)
posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)

# ---- Visualization ----
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(betas.numpy(), color='#E91E63')
axes[0].set_title(r'$\beta_t$ (noise schedule)', fontsize=13)
axes[0].set_xlabel('Timestep $t$')
axes[0].set_ylabel(r'$\beta_t$')

axes[1].plot(alphas_cumprod.numpy(), color='#1565C0')
axes[1].set_title(r'$\bar{\alpha}_t$ (cumulative signal retention)', fontsize=13)
axes[1].set_xlabel('Timestep $t$')
axes[1].set_ylabel(r'$\bar{\alpha}_t$')

axes[2].plot(sqrt_one_minus_alphas_cumprod.numpy(), color='#4CAF50')
axes[2].set_title(r'$\sqrt{1-\bar{\alpha}_t}$ (noise level)', fontsize=13)
axes[2].set_xlabel('Timestep $t$')
axes[2].set_ylabel(r'$\sqrt{1-\bar{\alpha}_t}$')

plt.tight_layout()
plt.show()

print(f"At t=0:   signal={sqrt_alphas_cumprod[0]:.4f}, noise={sqrt_one_minus_alphas_cumprod[0]:.4f}")
print(f"At t=149: signal={sqrt_alphas_cumprod[149]:.4f}, noise={sqrt_one_minus_alphas_cumprod[149]:.4f}")
print(f"At t=299: signal={sqrt_alphas_cumprod[299]:.4f}, noise={sqrt_one_minus_alphas_cumprod[299]:.4f}")

In [ ]:
# ============================================================
# Forward Diffusion (Adding Noise)
# ============================================================

def q_sample(x_0, t, noise=None):
    """Add noise to x_0 at timestep t (forward process).
    
    q(x_t | x_0) = N(sqrt(alpha_bar_t) * x_0, (1 - alpha_bar_t) * I)
    
    Args:
        x_0: Clean images, shape (B, C, H, W)
        t: Timesteps, shape (B,) with values in [0, T)
        noise: Optional pre-generated noise
    
    Returns:
        x_t: Noised images
        noise: The noise that was added
    """
    if noise is None:
        noise = torch.randn_like(x_0)
    
    sqrt_alpha = sqrt_alphas_cumprod[t].reshape(-1, 1, 1, 1)
    sqrt_one_minus_alpha = sqrt_one_minus_alphas_cumprod[t].reshape(-1, 1, 1, 1)
    
    return sqrt_alpha * x_0 + sqrt_one_minus_alpha * noise, noise


# ---- Visualize the forward process on a sample image ----
# Create a simple test phantom
def make_test_phantom(size=32):
    """Create a simple PET-like phantom for visualization."""
    img = np.zeros((size, size), dtype=np.float32)
    y, x = np.mgrid[-1:1:size*1j, -1:1:size*1j]
    
    # Body outline
    body = (x**2 / 0.7**2 + y**2 / 0.9**2) < 1.0
    img[body] = 0.2
    
    # Hot spot (tumor)
    tumor = ((x - 0.2)**2 + (y - 0.3)**2) < 0.08**2
    img[tumor] = 0.9
    
    # Ring structure (heart)
    r = np.sqrt((x + 0.15)**2 + (y + 0.1)**2)
    ring = (r > 0.15) & (r < 0.25)
    img[ring] = 0.7
    
    return img

phantom = make_test_phantom(32)
x_0 = torch.FloatTensor(phantom).unsqueeze(0).unsqueeze(0)  # (1, 1, 32, 32)

timesteps_to_show = [0, 50, 100, 200, 299]
fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(15, 3))

for i, t_val in enumerate(timesteps_to_show):
    t = torch.tensor([t_val])
    x_t, _ = q_sample(x_0, t)
    axes[i].imshow(x_t[0, 0].numpy(), cmap='hot', vmin=-2, vmax=2)
    axes[i].set_title(f't = {t_val}', fontsize=12)
    axes[i].axis('off')

fig.suptitle('Forward Diffusion Process: Clean Image → Pure Noise', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Generate Synthetic PET Training Data
# ============================================================

def generate_synthetic_pet_slices(n_samples=2000, image_size=32):
    """Generate synthetic 2D PET-like slices for training.
    
    Each slice contains:
    - Low background activity
    - Random Gaussian blobs (simulating tracer uptake regions)
    - Optional ring structures (simulating myocardium)
    
    All images are normalized to [0, 1].
    """
    slices = []
    for _ in range(n_samples):
        img = np.zeros((image_size, image_size), dtype=np.float32)
        y, x = np.mgrid[-1:1:image_size*1j, -1:1:image_size*1j]
        
        # Background activity
        img += 0.1
        
        # Random blobs (simulating uptake regions)
        n_blobs = np.random.randint(2, 6)
        for _ in range(n_blobs):
            cx, cy = np.random.uniform(-0.6, 0.6, 2)
            sigma = np.random.uniform(0.08, 0.25)
            intensity = np.random.uniform(0.3, 1.0)
            blob = intensity * np.exp(-((x - cx)**2 + (y - cy)**2) / (2 * sigma**2))
            img += blob
        
        # Optional ring structure (myocardium-like)
        if np.random.random() < 0.4:
            r = np.sqrt(x**2 + y**2)
            ring = (r > 0.2) & (r < 0.4)
            img[ring] += np.random.uniform(0.3, 0.7)
        
        # Normalize to [0, 1]
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        slices.append(img)
    
    return np.array(slices)


train_data = generate_synthetic_pet_slices(2000, image_size=32)
print(f"Training data shape: {train_data.shape}")
print(f"Value range: [{train_data.min():.3f}, {train_data.max():.3f}]")

# Visualize some training samples
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(train_data[i], cmap='hot', vmin=0, vmax=1)
    ax.axis('off')
fig.suptitle('Sample Synthetic PET Training Images', fontsize=14)
plt.tight_layout()
plt.show()

## Time-Conditioned U-Net

The denoising network $\epsilon_\theta(x_t, t)$ must know **what timestep** it is denoising from. The noise level at $t=10$ is very different from $t=290$.

**Time embedding**: We use sinusoidal position encoding (same idea as in Transformers), followed by a small MLP:

$$\text{PE}(t, 2i) = \sin\left(\frac{t}{10000^{2i/d}}\right), \quad \text{PE}(t, 2i+1) = \cos\left(\frac{t}{10000^{2i/d}}\right)$$

**Architecture**: A small U-Net with:
- **Encoder**: 2 downsampling blocks (32 $\rightarrow$ 16 $\rightarrow$ 8)
- **Bottleneck**: ConvBlock at 8$\times$8 resolution
- **Decoder**: 2 upsampling blocks with skip connections (8 $\rightarrow$ 16 $\rightarrow$ 32)
- **Time conditioning**: Time embedding is projected and added to feature maps in each ConvBlock
- **~100K parameters**: Small enough for CPU training in reasonable time

In [ ]:
# ============================================================
# Time Embedding + U-Net Architecture
# ============================================================

class SinusoidalPositionEmbedding(nn.Module):
    """Sinusoidal position embedding for timestep conditioning.
    
    Maps integer timestep t to a d-dimensional vector using sin/cos
    at different frequencies — same idea as Transformer positional encoding.
    """
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    
    def forward(self, t):
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t[:, None].float() * emb[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)


class ConvBlock(nn.Module):
    """Convolution block with time conditioning.
    
    Two convolutions with GroupNorm and SiLU activation.
    Time embedding is projected and added between convolutions.
    """
    def __init__(self, in_c, out_c, time_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, 1, 1)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, 1, 1)
        self.norm1 = nn.GroupNorm(4, out_c)
        self.norm2 = nn.GroupNorm(4, out_c)
        self.time_mlp = nn.Linear(time_dim, out_c)
        self.act = nn.SiLU()
    
    def forward(self, x, t_emb):
        h = self.act(self.norm1(self.conv1(x)))
        # Add time information
        h = h + self.time_mlp(t_emb)[:, :, None, None]
        h = self.act(self.norm2(self.conv2(h)))
        return h


class SimpleUNet(nn.Module):
    """Small U-Net for 32x32 images with time conditioning.
    
    Architecture:
        Encoder:  32x32 (base_dim) -> 16x16 (base_dim*2) -> 8x8 (base_dim*4)
        Decoder:  8x8 -> 16x16 (skip) -> 32x32 (skip) -> output
    
    Time embedding is injected into every ConvBlock.
    """
    
    def __init__(self, in_channels=1, time_dim=64, base_dim=32):
        super().__init__()
        
        # Time embedding network
        self.time_embed = nn.Sequential(
            SinusoidalPositionEmbedding(time_dim),
            nn.Linear(time_dim, time_dim * 2),
            nn.SiLU(),
            nn.Linear(time_dim * 2, time_dim),
        )
        
        # Encoder
        self.enc1 = ConvBlock(in_channels, base_dim, time_dim)      # 32x32
        self.down1 = nn.Conv2d(base_dim, base_dim, 4, 2, 1)        # 32->16
        self.enc2 = ConvBlock(base_dim, base_dim * 2, time_dim)     # 16x16
        self.down2 = nn.Conv2d(base_dim * 2, base_dim * 2, 4, 2, 1)  # 16->8
        
        # Bottleneck
        self.bottleneck = ConvBlock(base_dim * 2, base_dim * 4, time_dim)  # 8x8
        
        # Decoder
        self.up2 = nn.ConvTranspose2d(base_dim * 4, base_dim * 2, 4, 2, 1)  # 8->16
        self.dec2 = ConvBlock(base_dim * 4, base_dim * 2, time_dim)  # concat skip
        self.up1 = nn.ConvTranspose2d(base_dim * 2, base_dim, 4, 2, 1)     # 16->32
        self.dec1 = ConvBlock(base_dim * 2, base_dim, time_dim)      # concat skip
        
        # Output projection
        self.final = nn.Conv2d(base_dim, in_channels, 1)
    
    def forward(self, x, t):
        t_emb = self.time_embed(t)
        
        # Encode
        e1 = self.enc1(x, t_emb)          # (B, base_dim, 32, 32)
        e2 = self.enc2(self.down1(e1), t_emb)  # (B, base_dim*2, 16, 16)
        
        # Bottleneck
        b = self.bottleneck(self.down2(e2), t_emb)  # (B, base_dim*4, 8, 8)
        
        # Decode with skip connections
        d2 = self.dec2(torch.cat([self.up2(b), e2], dim=1), t_emb)  # (B, base_dim*2, 16, 16)
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1), t_emb)  # (B, base_dim, 32, 32)
        
        return self.final(d1)  # (B, 1, 32, 32)


# Instantiate and inspect
model = SimpleUNet(in_channels=1, time_dim=64, base_dim=32)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

# Verify forward pass
dummy_x = torch.randn(2, 1, 32, 32)
dummy_t = torch.tensor([10, 200])
dummy_out = model(dummy_x, dummy_t)
print(f"Input shape:  {dummy_x.shape}")
print(f"Output shape: {dummy_out.shape}")
assert dummy_out.shape == dummy_x.shape, "Output shape must match input shape!"

In [ ]:
# ============================================================
# Training Loop
# ============================================================
# ~10-20 min on CPU with 32x32 images and 2000 samples

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

# Move schedule tensors to device
betas_d = betas.to(device)
alphas_d = alphas.to(device)
alphas_cumprod_d = alphas_cumprod.to(device)
sqrt_alphas_cumprod_d = sqrt_alphas_cumprod.to(device)
sqrt_one_minus_alphas_cumprod_d = sqrt_one_minus_alphas_cumprod.to(device)

def q_sample_device(x_0, t, noise=None):
    """Forward process on the training device."""
    if noise is None:
        noise = torch.randn_like(x_0)
    sqrt_alpha = sqrt_alphas_cumprod_d[t].reshape(-1, 1, 1, 1)
    sqrt_one_minus_alpha = sqrt_one_minus_alphas_cumprod_d[t].reshape(-1, 1, 1, 1)
    return sqrt_alpha * x_0 + sqrt_one_minus_alpha * noise, noise

model = SimpleUNet(in_channels=1, time_dim=64, base_dim=32).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Create dataset and dataloader
dataset = torch.utils.data.TensorDataset(
    torch.FloatTensor(train_data).unsqueeze(1)  # (N, 1, 32, 32)
)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)

losses = []
n_epochs = 40

print(f"Starting training: {n_epochs} epochs, {len(dataloader)} batches/epoch")
print(f"Dataset: {len(dataset)} samples, batch size: 64")
print("-" * 50)

import time
start_time = time.time()

for epoch in range(n_epochs):
    epoch_loss = 0.0
    model.train()
    
    for (batch,) in dataloader:
        batch = batch.to(device)
        
        # Sample random timesteps for each image in batch
        t = torch.randint(0, T, (batch.size(0),), device=device)
        
        # Forward process: add noise
        noise = torch.randn_like(batch)
        x_t, _ = q_sample_device(batch, t, noise)
        
        # Predict the noise
        predicted_noise = model(x_t, t)
        loss = F.mse_loss(predicted_noise, noise)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        elapsed = time.time() - start_time
        print(f"Epoch {epoch+1:3d}/{n_epochs}, Loss: {avg_loss:.6f}, "
              f"Elapsed: {elapsed:.0f}s")

total_time = time.time() - start_time
print(f"\nTraining complete in {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"Final loss: {losses[-1]:.6f}")

In [ ]:
# ============================================================
# Plot Training Loss
# ============================================================

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(losses) + 1), losses, color='#E91E63', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MSE Loss', fontsize=12)
ax.set_title('Diffusion Model Training Loss', fontsize=14)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print(f"Loss decreased from {losses[0]:.6f} to {losses[-1]:.6f} "
      f"({(1 - losses[-1]/losses[0])*100:.1f}% reduction)")

## Sampling (Reverse Process)

To generate an image, start from pure noise $x_T \sim \mathcal{N}(0, \mathbf{I})$ and iteratively denoise:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}} \, \epsilon_\theta(x_t, t) \right) + \sigma_t \, z$$

where:
- $z \sim \mathcal{N}(0, \mathbf{I})$ for $t > 1$, and $z = 0$ for $t = 0$
- $\sigma_t = \sqrt{\beta_t}$ (simplified variance)

Each call to the model removes a small amount of noise. After $T$ steps, we get a clean sample from the learned distribution.

In [ ]:
# ============================================================
# Sampling Functions
# ============================================================

@torch.no_grad()
def p_sample(model, x_t, t_idx):
    """Single reverse diffusion step: x_t -> x_{t-1}."""
    t = torch.full((x_t.size(0),), t_idx, device=x_t.device, dtype=torch.long)
    
    beta = betas[t_idx]
    alpha = alphas[t_idx]
    alpha_bar = alphas_cumprod[t_idx]
    
    # Predict noise
    predicted_noise = model(x_t, t)
    
    # Compute mean of p(x_{t-1} | x_t)
    mean = (1.0 / torch.sqrt(alpha)) * (
        x_t - (beta / torch.sqrt(1.0 - alpha_bar)) * predicted_noise
    )
    
    if t_idx > 0:
        noise = torch.randn_like(x_t)
        sigma = torch.sqrt(beta)
        return mean + sigma * noise
    else:
        return mean  # No noise at final step


@torch.no_grad()
def sample(model, n_samples=16, image_size=32):
    """Generate samples via full reverse diffusion (T steps)."""
    model.eval()
    x = torch.randn(n_samples, 1, image_size, image_size, device=device)
    
    for t in reversed(range(T)):
        x = p_sample(model, x, t)
    
    return x.cpu()


# Generate samples
print("Generating 16 samples via reverse diffusion...")
samples = sample(model, n_samples=16)
print(f"Generated samples shape: {samples.shape}")
print(f"Value range: [{samples.min():.3f}, {samples.max():.3f}]")

# Visualize in a 4x4 grid
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    img = samples[i, 0].numpy()
    ax.imshow(img, cmap='hot', vmin=0, vmax=1)
    ax.axis('off')
fig.suptitle('Unconditional Samples from Trained Diffusion Model', fontsize=14)
plt.tight_layout()
plt.show()

## Measurement-Conditioned Sampling (Posterior Guidance)

For **reconstruction**, we don't want unconditional samples from $p(x)$. We want samples from the **posterior** $p(x | y)$, where $y$ are our measurements.

Given a forward model $y = \mathcal{A}(x) + \text{noise}$, we can guide the reverse process toward measurement consistency:

$$x_{t-1} = \underbrace{\text{denoise}(x_t)}_{\text{prior}} - \lambda \underbrace{\nabla_x \| \mathcal{A}(\hat{x}_0) - y \|^2}_{\text{data fidelity}}$$

where $\hat{x}_0$ is the predicted clean image from $x_t$:

$$\hat{x}_0 = \frac{x_t - \sqrt{1-\bar{\alpha}_t}\, \epsilon_\theta(x_t, t)}{\sqrt{\bar{\alpha}_t}}$$

**Key insight**: This combines the learned prior (diffusion model) with the physics constraint (measurement model) **without retraining**. The same diffusion model works for different:
- Noise levels
- Numbers of projection angles
- Scanner geometries

This is sometimes called **Diffusion Posterior Sampling (DPS)** or **measurement-guided diffusion**.

In [ ]:
# ============================================================
# Measurement-Consistent Sampling
# ============================================================

try:
    from skimage.transform import radon, iradon
    HAS_SKIMAGE = True
    print("scikit-image available: using Radon transform for forward model.")
except ImportError:
    HAS_SKIMAGE = False
    print("scikit-image not available: using simplified random projection forward model.")


def create_forward_operator(image_size=32, n_angles=30):
    """Create a forward projection operator.
    
    If scikit-image is available, uses the Radon transform (sinogram).
    Otherwise, uses a random Gaussian measurement matrix as a proxy.
    
    Returns:
        forward_fn: function mapping image (H, W) -> measurements
        backward_fn: function mapping measurements -> image (H, W) [adjoint/pseudoinverse]
        theta: projection angles (or None)
    """
    theta = np.linspace(0, 180, n_angles, endpoint=False)
    
    if HAS_SKIMAGE:
        def forward_fn(x_np):
            return radon(x_np, theta=theta)
        
        def backward_fn(sino, output_size=image_size):
            return iradon(sino, theta=theta, filter_name=None,
                         output_size=output_size)
        
        return forward_fn, backward_fn, theta
    else:
        # Fallback: random Gaussian measurements
        n_measurements = n_angles * image_size  # comparable to sinogram size
        np.random.seed(123)
        A = np.random.randn(n_measurements, image_size * image_size).astype(np.float32)
        A /= np.sqrt(n_measurements)  # normalize
        A_pinv = np.linalg.pinv(A)
        
        def forward_fn(x_np):
            return A @ x_np.flatten()
        
        def backward_fn(y, output_size=image_size):
            return (A_pinv @ y).reshape(output_size, output_size)
        
        return forward_fn, backward_fn, theta


@torch.no_grad()
def guided_sample(model, measurements, forward_fn, backward_fn,
                  n_samples=4, guidance_scale=0.3, image_size=32):
    """Sample with measurement consistency guidance (DPS-style).
    
    At every few reverse steps, nudge the reconstruction toward
    consistency with the observed measurements.
    
    Args:
        model: Trained diffusion model
        measurements: Observed measurements (sinogram or projections)
        forward_fn: Forward operator A: image -> measurements
        backward_fn: Adjoint/pseudoinverse: measurements -> image
        n_samples: Number of posterior samples to draw
        guidance_scale: Strength of measurement guidance
        image_size: Spatial size of images
    
    Returns:
        Tensor of shape (n_samples, 1, H, W)
    """
    model.eval()
    all_samples = []
    
    for s in range(n_samples):
        x = torch.randn(1, 1, image_size, image_size, device=device)
        
        for t in reversed(range(T)):
            # Standard reverse diffusion step
            x_denoised = p_sample(model, x, t)
            
            # Measurement consistency correction (every 10 steps)
            if t % 10 == 0 and t > 0:
                x_np = x_denoised[0, 0].cpu().numpy()
                
                # Forward project current estimate
                projected = forward_fn(x_np)
                
                # Compute residual in measurement space
                residual = measurements - projected
                
                # Back-project residual (gradient direction)
                correction = backward_fn(residual, output_size=image_size)
                
                # Normalize correction
                max_abs = np.max(np.abs(correction)) + 1e-8
                correction = correction / max_abs
                
                # Apply correction
                correction_tensor = torch.FloatTensor(correction).unsqueeze(0).unsqueeze(0).to(device)
                # Scale guidance by noise level (less correction as we approach t=0)
                noise_level = sqrt_one_minus_alphas_cumprod[t].item()
                x_denoised = x_denoised + guidance_scale * noise_level * correction_tensor
            
            x = x_denoised
        
        all_samples.append(x.cpu())
        if (s + 1) % 2 == 0:
            print(f"  Generated sample {s+1}/{n_samples}")
    
    return torch.cat(all_samples, dim=0)


# ---- Demo: Reconstruct a test phantom ----
print("Creating forward operator...")
forward_fn, backward_fn, theta = create_forward_operator(image_size=32, n_angles=30)

# Ground truth phantom
gt_phantom = make_test_phantom(32)

# Generate noisy measurements
clean_measurements = forward_fn(gt_phantom)
noise_level_meas = 0.05 * np.max(np.abs(clean_measurements))
noisy_measurements = clean_measurements + noise_level_meas * np.random.randn(*clean_measurements.shape)

print(f"Measurement shape: {noisy_measurements.shape}")
print(f"\nRunning guided sampling (4 posterior samples)...")
guided_samples = guided_sample(
    model, noisy_measurements, forward_fn, backward_fn,
    n_samples=4, guidance_scale=0.3, image_size=32
)

# Visualize
fig, axes = plt.subplots(1, 6, figsize=(18, 3))

axes[0].imshow(gt_phantom, cmap='hot', vmin=0, vmax=1)
axes[0].set_title('Ground Truth')
axes[0].axis('off')

# Simple back-projection baseline
bp_recon = backward_fn(noisy_measurements, output_size=32)
bp_recon = np.clip((bp_recon - bp_recon.min()) / (bp_recon.max() - bp_recon.min() + 1e-8), 0, 1)
axes[1].imshow(bp_recon, cmap='hot', vmin=0, vmax=1)
axes[1].set_title('Back-projection')
axes[1].axis('off')

for i in range(4):
    img = guided_samples[i, 0].numpy()
    img = np.clip(img, 0, 1)
    axes[i + 2].imshow(img, cmap='hot', vmin=0, vmax=1)
    axes[i + 2].set_title(f'Diffusion #{i+1}')
    axes[i + 2].axis('off')

fig.suptitle('Measurement-Guided Diffusion Reconstruction', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Comparison: OSEM vs GAN vs Diffusion
# ============================================================

def compute_psnr(img, ref):
    """Peak Signal-to-Noise Ratio."""
    mse = np.mean((img - ref) ** 2)
    if mse < 1e-10:
        return float('inf')
    return 10 * np.log10(1.0 / mse)

def compute_ssim(img, ref):
    """Simplified SSIM (structural similarity) for single-channel images."""
    C1 = (0.01) ** 2
    C2 = (0.03) ** 2
    
    mu_x = np.mean(img)
    mu_y = np.mean(ref)
    sigma_x = np.std(img)
    sigma_y = np.std(ref)
    sigma_xy = np.mean((img - mu_x) * (ref - mu_y))
    
    ssim = ((2 * mu_x * mu_y + C1) * (2 * sigma_xy + C2)) / \
           ((mu_x**2 + mu_y**2 + C1) * (sigma_x**2 + sigma_y**2 + C2))
    return ssim

def compute_nmse(img, ref):
    """Normalized Mean Squared Error."""
    return np.sum((img - ref) ** 2) / np.sum(ref ** 2)


# ---- OSEM-like reconstruction (simplified iterative) ----
def simple_iterative_recon(measurements, forward_fn, backward_fn,
                           n_iters=20, image_size=32):
    """Simplified iterative reconstruction (OSEM-like).
    
    This is a basic MLEM-style update as a stand-in for the full OSEM
    from Notebook II. For the real implementation, see 02_osem_reconstruction.ipynb.
    """
    # Initialize with back-projection
    recon = backward_fn(measurements, output_size=image_size)
    recon = np.clip(recon, 0.01, None)  # Ensure positivity
    recon = recon / (recon.max() + 1e-8)
    
    for i in range(n_iters):
        # Forward project current estimate
        projected = forward_fn(recon)
        
        # Ratio of measured to projected
        ratio = measurements / (projected + 1e-8)
        ratio = np.clip(ratio, 0, 10)  # Stabilize
        
        # Back-project ratio
        correction = backward_fn(ratio, output_size=image_size)
        
        # Sensitivity image (normalization)
        ones_proj = forward_fn(np.ones((image_size, image_size)))
        sensitivity = backward_fn(ones_proj, output_size=image_size)
        sensitivity = np.clip(sensitivity, 1e-8, None)
        
        # MLEM update
        recon = recon * correction / sensitivity
        recon = np.clip(recon, 0, None)
    
    # Normalize
    recon = (recon - recon.min()) / (recon.max() - recon.min() + 1e-8)
    return recon


# Run OSEM-like reconstruction
print("Running iterative (OSEM-like) reconstruction...")
osem_recon = simple_iterative_recon(
    noisy_measurements, forward_fn, backward_fn, n_iters=20, image_size=32
)

# Diffusion reconstruction: use mean of guided samples
diffusion_recon = guided_samples.mean(dim=0)[0].numpy()
diffusion_recon = np.clip(diffusion_recon, 0, 1)

# GAN reconstruction placeholder
# In practice, you would load the trained generator from Notebook III-b
# For this comparison, we simulate a GAN-like result by applying
# a learned denoising step to the back-projection
print("Note: GAN result is a simulated placeholder.")
print("For a real GAN reconstruction, see Notebook III-b.")
gan_placeholder = bp_recon.copy()
# Simulate GAN output: sharpen and denoise the back-projection
from scipy.ndimage import gaussian_filter
gan_placeholder = gaussian_filter(gan_placeholder, sigma=0.8)
gan_placeholder = np.clip(gan_placeholder, 0, 1)

# ---- Visual Comparison ----
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
methods = [
    ('Ground Truth', gt_phantom),
    ('OSEM (iterative)', osem_recon),
    ('GAN (placeholder)', gan_placeholder),
    ('Diffusion (mean)', diffusion_recon),
]

for ax, (title, img) in zip(axes, methods):
    ax.imshow(img, cmap='hot', vmin=0, vmax=1)
    ax.set_title(title, fontsize=12)
    ax.axis('off')

fig.suptitle('Reconstruction Comparison', fontsize=14)
plt.tight_layout()
plt.show()

# ---- Metrics Table ----
print("\n" + "=" * 55)
print(f"{'Method':<25} {'PSNR (dB)':>10} {'SSIM':>8} {'NMSE':>10}")
print("=" * 55)

for name, recon in [('OSEM (iterative)', osem_recon),
                     ('GAN (placeholder)', gan_placeholder),
                     ('Diffusion (mean)', diffusion_recon)]:
    psnr = compute_psnr(recon, gt_phantom)
    ssim = compute_ssim(recon, gt_phantom)
    nmse = compute_nmse(recon, gt_phantom)
    print(f"{name:<25} {psnr:>10.2f} {ssim:>8.4f} {nmse:>10.6f}")

print("=" * 55)
print("\nNote: GAN results are simulated. Real GAN performance depends on")
print("training quality — see Notebook III-b for actual GAN reconstruction.")

## Uncertainty Quantification

This is the **killer feature** of diffusion models for medical imaging.

Because the reverse process is stochastic, running guided sampling $N$ times produces $N$ **different** reconstructions — all consistent with the measurements, but varying in regions where the data is ambiguous.

**Recipe:**
1. Run guided sampling $N$ times (e.g., $N=8$)
2. Compute pixel-wise **mean** $\rightarrow$ best estimate
3. Compute pixel-wise **standard deviation** $\rightarrow$ uncertainty map

**Clinical value:**
- High uncertainty regions = model is unsure = radiologist should look more carefully
- Low uncertainty regions = consistent across samples = trustworthy
- This is **free** with diffusion — no extra training, no extra architecture
- **Impossible with OSEM** (deterministic) and **very hard with GANs** (would need dropout-based MC, which is unreliable)

In [ ]:
# ============================================================
# Uncertainty Maps from Multiple Posterior Samples
# ============================================================

N_UNCERTAINTY_SAMPLES = 8

print(f"Generating {N_UNCERTAINTY_SAMPLES} posterior samples for uncertainty estimation...")
print("(This may take a few minutes on CPU)")

uncertainty_samples = guided_sample(
    model, noisy_measurements, forward_fn, backward_fn,
    n_samples=N_UNCERTAINTY_SAMPLES, guidance_scale=0.3, image_size=32
)

# Compute statistics
samples_np = uncertainty_samples[:, 0].numpy()  # (N, 32, 32)
samples_np = np.clip(samples_np, 0, 1)

mean_recon = samples_np.mean(axis=0)
std_recon = samples_np.std(axis=0)

print(f"\nMean reconstruction range: [{mean_recon.min():.3f}, {mean_recon.max():.3f}]")
print(f"Std (uncertainty) range:   [{std_recon.min():.4f}, {std_recon.max():.4f}]")

# ---- Visualization ----
fig, axes = plt.subplots(2, 5, figsize=(18, 8))

# Top row: individual samples
for i in range(5):
    axes[0, i].imshow(samples_np[i], cmap='hot', vmin=0, vmax=1)
    axes[0, i].set_title(f'Sample {i+1}', fontsize=11)
    axes[0, i].axis('off')

# Bottom row: ground truth, mean, std, absolute error, error vs uncertainty
axes[1, 0].imshow(gt_phantom, cmap='hot', vmin=0, vmax=1)
axes[1, 0].set_title('Ground Truth', fontsize=11)
axes[1, 0].axis('off')

axes[1, 1].imshow(mean_recon, cmap='hot', vmin=0, vmax=1)
axes[1, 1].set_title('Mean (Best Estimate)', fontsize=11)
axes[1, 1].axis('off')

im_std = axes[1, 2].imshow(std_recon, cmap='viridis')
axes[1, 2].set_title('Uncertainty (Std)', fontsize=11)
axes[1, 2].axis('off')
plt.colorbar(im_std, ax=axes[1, 2], fraction=0.046)

abs_error = np.abs(mean_recon - gt_phantom)
im_err = axes[1, 3].imshow(abs_error, cmap='magma')
axes[1, 3].set_title('Absolute Error', fontsize=11)
axes[1, 3].axis('off')
plt.colorbar(im_err, ax=axes[1, 3], fraction=0.046)

# Scatter plot: uncertainty vs error
axes[1, 4].scatter(std_recon.flatten(), abs_error.flatten(),
                   alpha=0.3, s=5, color='#1565C0')
axes[1, 4].set_xlabel('Uncertainty (std)', fontsize=10)
axes[1, 4].set_ylabel('Absolute error', fontsize=10)
axes[1, 4].set_title('Calibration', fontsize=11)

# Compute correlation
corr = np.corrcoef(std_recon.flatten(), abs_error.flatten())[0, 1]
axes[1, 4].text(0.05, 0.95, f'r = {corr:.3f}',
                transform=axes[1, 4].transAxes, fontsize=10,
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

fig.suptitle('Uncertainty Quantification from Diffusion Posterior Samples', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nCorrelation between uncertainty and error: {corr:.3f}")
print("A positive correlation means the model is well-calibrated:")
print("  High uncertainty <-> High error (model knows what it doesn't know)")

## SOTA Survey: Diffusion Models in Medical Image Reconstruction

The field is moving rapidly. Here are the key developments:

### Foundational Methods

| Paper | Key Idea |
|-------|----------|
| **Score-SDE** (Song et al., 2021) | Unified framework: score-based models + diffusion = same thing. Score matching for inverse problems. |
| **DPS** (Chung et al., 2023) | Diffusion Posterior Sampling: approximate $\nabla \log p(y|x_t)$ without retraining. What we implemented above. |
| **DiffusionMBIR** (Chung et al., 2023) | Combine diffusion prior with model-based iterative reconstruction (MBIR). Physics + learning. |
| **MCG** (Chung et al., 2022) | Manifold-constrained gradient: project onto measurement-consistent manifold at each step. |

### Posterior Mean Models
- Instead of sampling from $p(x|y)$, directly estimate $\mathbb{E}[x|y]$
- Faster (single forward pass) but loses uncertainty quantification
- Useful when speed matters more than uncertainty

### Speeding Up Sampling
- **DDIM** (Song et al., 2021): Deterministic sampling, skip steps (300 $\rightarrow$ 50)
- **Consistency Models** (Song et al., 2023): Single-step generation via consistency training
- **Progressive Distillation**: Train a student to do in 2 steps what the teacher does in 1000

### 3D Challenges
- 3D medical volumes are huge: 256$\times$256$\times$256 $\times$ 300 steps = enormous memory
- **Latent Diffusion** (Rombach et al., 2022): Compress to latent space, diffuse there, decode back
- **Slice-by-slice**: Apply 2D diffusion to each slice (loses 3D consistency)
- **Patch-based**: Diffuse on overlapping patches and stitch

### Current Limitations
1. **Sampling speed**: 300+ forward passes per reconstruction (minutes vs. milliseconds for GANs)
2. **Memory**: 3D diffusion requires very large GPU memory
3. **Guidance design**: Different inverse problems need different guidance strategies
4. **Evaluation**: Hard to evaluate sample diversity and calibration in medical settings

## Summary

### What We Covered

1. **DDPM fundamentals**: Forward process (add noise) + reverse process (learn to denoise)
2. **Time-conditioned U-Net**: Sinusoidal time embedding injected into a small encoder-decoder
3. **Training**: Simple noise-prediction objective $\|\epsilon - \epsilon_\theta(x_t, t)\|^2$
4. **Unconditional sampling**: Start from noise, iteratively denoise over $T$ steps
5. **Measurement-guided sampling**: Combine learned prior with physics constraints at inference time
6. **Uncertainty quantification**: Multiple posterior samples $\rightarrow$ pixel-wise mean and variance

### Diffusion vs. Other Approaches

| Aspect | OSEM | GAN | Diffusion |
|--------|------|-----|-----------|
| Training | None | Adversarial (unstable) | Denoising (stable) |
| Mode collapse | N/A | Yes | No |
| Uncertainty | No | Difficult | Free |
| Inference speed | Fast (iterative) | Very fast | Slow (many steps) |
| Image quality | Noisy at low counts | Sharp but may hallucinate | High quality |
| Physics integration | Native | Post-hoc | Via guidance |

### Tutorial Series Recap

- **Notebook I**: Data loading and preprocessing (DICOM, sinograms, normalization)
- **Notebook II**: OSEM reconstruction from scratch (physics-based, iterative)
- **Notebook III-a**: Deep learning basics for medical imaging
- **Notebook III-b**: GAN-based reconstruction (adversarial training, conditional generation)
- **Notebook III-c**: Diffusion-based reconstruction (this notebook) — principled probabilistic framework with uncertainty quantification

### Looking Ahead

Diffusion models are likely the **future of medical image reconstruction**. The combination of:
- Stable training
- No mode collapse
- Built-in uncertainty quantification
- Flexible conditioning via guidance

...makes them ideal for clinical applications where reliability and interpretability matter. The main barrier — sampling speed — is being rapidly addressed by consistency models, DDIM, and distillation techniques.